# Train Dwell Time Prediction — Modeling

This notebook is the final step of the pipeline. It loads the feature-engineered dataset produced by notebook 04 and trains multiple regression models to predict train dwell time at the MINOT station.

**Data:** `../data/features_engineered.csv` (output of notebook 04, not versioned — proprietary)

**Target:** `DWELL_TIME` (in hours)

**Models tested:**
1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. ElasticNet Regression
5. Random Forest Regressor
6. Gradient Boosting Regressor
7. XGBoost Regressor

**Evaluation metrics:** MAE, RMSE, R²

**Train/test split:** temporal — training = records with TA < 2024-01-01, test = records with TA ≥ 2024-01-01 (same split as notebook 02)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print("XGBoost available")
except ImportError:
    print("XGBoost not installed — skipping")
    XGBOOST_AVAILABLE = False

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_palette('husl')

## 1. Load Data

Load the processed dataset from notebook 04. Outlier flags are already computed (notebook 02 logic), so we just need to filter `IS_OUTLIER == 0`.

In [ ]:
df = pd.read_csv('../data/features_engineered.csv')

print(f"Raw shape: {df.shape}")
print(f"Outlier breakdown:\n{df['IS_OUTLIER'].value_counts()}")

# Keep only non-outlier rows (as determined in notebook 02)
df = df[df['IS_OUTLIER'] == 0].copy()
df.reset_index(drop=True, inplace=True)

df['TA'] = pd.to_datetime(df['TA'], utc=True)
df['TD'] = pd.to_datetime(df['TD'], utc=True)

print(f"\nShape after outlier removal: {df.shape}")
print(f"Date range: {df['TA'].min()} → {df['TA'].max()}")
print(f"\nDWELL_TIME stats:")
print(df['DWELL_TIME'].describe())

In [ ]:
df.head()

## 2. Feature Selection

All features were engineered in notebook 04. Here we just select the columns to pass to the models.

Features fall into 3 groups:
- **Temporal:** Hour of Day, Day of Week, Month, Day Type, Is Holiday
- **Operational:** Inspection Requirement, Train Priority, Mainline flag
- **Congestion:** AllTrains, MainlineTrains, YardTrains, PriorityTrains, PriorityMainline, PriorityYard

In [ ]:
# Columns to drop (non-features)
drop_cols = ['TA', 'TD', 'DWELL_TIME', 'IS_OUTLIER', 'DWELL_TIME_BIN',
             'TRAIN_ID', 'DESTINATION', 'ArrivalDestination', 'DATE',
             'REQ_INSP', 'STN_TYPE_CD', 'DPT_DIR', 'LAST_CREW_STATION',
             'SECOND_LAST_CREW_STATION', 'case', 'TRAVEL_TIME',
             'TRN_MILES_TOT_TD', 'TRN_MILES_TOT_DEST', 'STN_SEQ_NBR_DEST', 'DISTANCE']

# Only drop columns that actually exist
drop_cols = [c for c in drop_cols if c in df.columns]

feature_cols = [c for c in df.columns if c not in drop_cols + ['DWELL_TIME', 'TA', 'TD']]

# Also drop any remaining object columns (shouldn't be any after NB04 encoding)
object_cols = df[feature_cols].select_dtypes(include='object').columns.tolist()
if object_cols:
    print(f"Dropping remaining object columns: {object_cols}")
    feature_cols = [c for c in feature_cols if c not in object_cols]

X = df[feature_cols].copy()
y = df['DWELL_TIME'].copy()

print(f"Feature matrix: {X.shape}")
print(f"\nFeatures ({len(feature_cols)}):")
print(feature_cols)

## 3. Train / Test Split

Same temporal split as notebook 02: training data = before January 1, 2024. This avoids data leakage since we're predicting future dwell times.

In [ ]:
split_date = pd.Timestamp('2024-01-01', tz='UTC')

train_mask = df['TA'] < split_date
test_mask  = df['TA'] >= split_date

X_train = X[train_mask].copy()
X_test  = X[test_mask].copy()
y_train = y[train_mask].copy()
y_test  = y[test_mask].copy()

print(f"Training set : {X_train.shape[0]} samples  ({df[train_mask]['TA'].min().date()} → {df[train_mask]['TA'].max().date()})")
print(f"Test set     : {X_test.shape[0]} samples  ({df[test_mask]['TA'].min().date()} → {df[test_mask]['TA'].max().date()})")
print(f"\nTrain target stats: mean={y_train.mean():.3f}h, std={y_train.std():.3f}h")
print(f"Test  target stats: mean={y_test.mean():.3f}h, std={y_test.std():.3f}h")

## 4. Feature Scaling

Linear models require standardized features. Tree-based models don't, so we'll keep two versions.

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)
print("Scaling done.")

## 5. Model Training

Helper function to evaluate any model.

In [ ]:
def evaluate(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"{label:30s}  MAE={mae:.4f}h  RMSE={rmse:.4f}h  R²={r2:.4f}")
    return {'label': label, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

results = []  # collect test results for comparison

### 5.1 Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

evaluate(y_train, lr.predict(X_train_scaled), 'Linear Regression (train)')
res = evaluate(y_test, lr.predict(X_test_scaled), 'Linear Regression (test)')
results.append(('Linear Regression', res))

### 5.2 Ridge Regression

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

evaluate(y_train, ridge.predict(X_train_scaled), 'Ridge (train)')
res = evaluate(y_test, ridge.predict(X_test_scaled), 'Ridge (test)')
results.append(('Ridge', res))

### 5.3 Lasso Regression

In [ ]:
lasso = Lasso(alpha=0.01, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

evaluate(y_train, lasso.predict(X_train_scaled), 'Lasso (train)')
res = evaluate(y_test, lasso.predict(X_test_scaled), 'Lasso (test)')
results.append(('Lasso', res))

# Check how many features Lasso zeroed out
n_zero = np.sum(lasso.coef_ == 0)
print(f"\nLasso zeroed out {n_zero}/{len(lasso.coef_)} features")

### 5.4 ElasticNet Regression

In [ ]:
# alpha very small → mostly keeps the Lasso effect (l1_ratio=1.0)
en = ElasticNet(alpha=0.00001, l1_ratio=1.0, max_iter=10000)
en.fit(X_train_scaled, y_train)

evaluate(y_train, en.predict(X_train_scaled), 'ElasticNet (train)')
res = evaluate(y_test, en.predict(X_test_scaled), 'ElasticNet (test)')
results.append(('ElasticNet', res))

### 5.5 Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)  # no scaling needed for tree-based

evaluate(y_train, rf.predict(X_train), 'Random Forest (train)')
res = evaluate(y_test, rf.predict(X_test), 'Random Forest (test)')
results.append(('Random Forest', res))

### 5.6 Gradient Boosting

In [ ]:
gb = GradientBoostingRegressor(
    n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42
)
gb.fit(X_train, y_train)

evaluate(y_train, gb.predict(X_train), 'Gradient Boosting (train)')
res = evaluate(y_test, gb.predict(X_test), 'Gradient Boosting (test)')
results.append(('Gradient Boosting', res))

### 5.7 XGBoost

In [ ]:
if XGBOOST_AVAILABLE:
    xgb_model = xgb.XGBRegressor(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=42, n_jobs=-1, verbosity=0
    )
    xgb_model.fit(X_train, y_train)

    evaluate(y_train, xgb_model.predict(X_train), 'XGBoost (train)')
    res = evaluate(y_test, xgb_model.predict(X_test), 'XGBoost (test)')
    results.append(('XGBoost', res))
else:
    print("XGBoost skipped")

## 6. Model Comparison

In [ ]:
results_df = pd.DataFrame(
    [(name, r['MAE'], r['RMSE'], r['R2']) for name, r in results],
    columns=['Model', 'MAE', 'RMSE', 'R2']
).sort_values('R2', ascending=False).reset_index(drop=True)

print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric, color, title in zip(
    axes,
    ['MAE', 'RMSE', 'R2'],
    ['steelblue', 'salmon', 'mediumseagreen'],
    ['MAE (hours)', 'RMSE (hours)', 'R² Score']
):
    ax.barh(results_df['Model'], results_df[metric], color=color)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel(metric)
    ax.grid(axis='x', alpha=0.3)
    if metric == 'R2':
        ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)

plt.suptitle('Model Comparison — Test Set', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Best Model — Predictions vs Actual

In [ ]:
best_name = results_df.iloc[0]['Model']
print(f"Best model: {best_name}")

# Map name back to model object
_model_map = {
    'Linear Regression': (lr, X_test_scaled),
    'Ridge': (ridge, X_test_scaled),
    'Lasso': (lasso, X_test_scaled),
    'ElasticNet': (en, X_test_scaled),
    'Random Forest': (rf, X_test),
    'Gradient Boosting': (gb, X_test),
}
if XGBOOST_AVAILABLE:
    _model_map['XGBoost'] = (xgb_model, X_test)

best_model, best_X_test = _model_map[best_name]
y_pred_best = best_model.predict(best_X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted vs actual
axes[0].scatter(y_test, y_pred_best, alpha=0.6, s=40, color='steelblue')
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
axes[0].plot(lims, lims, 'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual DWELL_TIME (h)', fontsize=12)
axes[0].set_ylabel('Predicted DWELL_TIME (h)', fontsize=12)
axes[0].set_title(f'Predicted vs Actual — {best_name}', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Residuals
axes[1].scatter(y_pred_best, residuals, alpha=0.6, s=40, color='salmon')
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted DWELL_TIME (h)', fontsize=12)
axes[1].set_ylabel('Residual (actual − predicted)', fontsize=12)
axes[1].set_title(f'Residual Plot — {best_name}', fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nResiduals: mean={residuals.mean():.4f}h  std={residuals.std():.4f}h")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals, bins=25, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(x=0, color='red', linestyle='--', lw=2)
axes[0].set_xlabel('Residual (hours)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Residual Distribution', fontsize=13, fontweight='bold')
axes[0].grid(alpha=0.3)

stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot (Normality Check)', fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Feature Importance

Compare feature importances across tree-based models.

In [ ]:
importance_df = pd.DataFrame({'feature': X_train.columns})
importance_df['Random Forest'] = rf.feature_importances_
importance_df['Gradient Boosting'] = gb.feature_importances_

if XGBOOST_AVAILABLE:
    importance_df['XGBoost'] = xgb_model.feature_importances_
    importance_df['avg'] = importance_df[['Random Forest', 'Gradient Boosting', 'XGBoost']].mean(axis=1)
else:
    importance_df['avg'] = importance_df[['Random Forest', 'Gradient Boosting']].mean(axis=1)

importance_df = importance_df.sort_values('avg', ascending=False).head(15)

print("Top 15 features (average importance across tree models):")
print(importance_df[['feature', 'avg']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

x = np.arange(len(importance_df))
w = 0.25 if (XGBOOST_AVAILABLE and 'XGBoost' in importance_df.columns) else 0.35

ax.barh(x - w, importance_df['Random Forest'], w, label='Random Forest', color='steelblue')
ax.barh(x,     importance_df['Gradient Boosting'], w, label='Gradient Boosting', color='salmon')
if XGBOOST_AVAILABLE and 'XGBoost' in importance_df.columns:
    ax.barh(x + w, importance_df['XGBoost'], w, label='XGBoost', color='mediumseagreen')

ax.set_yticks(x)
ax.set_yticklabels(importance_df['feature'])
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('Top 15 Feature Importances — Tree-Based Models', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Hyperparameter Tuning (optional)

Commented out — ran out of time during the internship. Left here as a starting point for future work.

In [ ]:
# TODO: run this properly when we have more time / compute

# param_grid_rf = {
#     'n_estimators': [100, 200, 300],
#     'max_depth': [5, 10, 15, None],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4]
# }
#
# grid_rf = GridSearchCV(
#     RandomForestRegressor(random_state=42, n_jobs=-1),
#     param_grid_rf,
#     cv=5,
#     scoring='neg_root_mean_squared_error',
#     n_jobs=-1,
#     verbose=2
# )
# grid_rf.fit(X_train, y_train)
#
# print(f"Best params: {grid_rf.best_params_}")
# best_rf_tuned = grid_rf.best_estimator_
# evaluate(y_test, best_rf_tuned.predict(X_test), 'RF Tuned (test)')